In [ ]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np
import pandas as pd
import pyreadstat

from argus.utils.PathManager import PathManager_RES_Folder_UPOD

# Load preprocessed data

In [ ]:
data_folder = os.path.join(os.getcwd(), "..", "data")

# Load preprocessed training data (output of the training pipeline)
df_female = pd.read_parquet(os.path.join(data_folder, "female_argus_data.parquet"))

df_male = pd.read_parquet(os.path.join(data_folder, "male_argus_data.parquet"))

df = pd.concat([df_female, df_male])

df["index_dt"] = df["index_dt"].dt.date

df.info()

# Load data required for label logic

### Load OR interventions (OK verrichtingen)

In [ ]:
path_manager = PathManager_RES_Folder_UPOD()
path_verr_ok = path_manager.get_file_path("ARGUS25", "20250908", "verr_ok_20250527.sas7bdat")

In [ ]:
# Load data on interventions
df_verr_ok = pyreadstat.read_sas7bdat(path_verr_ok)[0]
df_verr_ok["studyId_0888"] = df_verr_ok["studyId_0888"].astype(int).astype(str)
df_verr_ok["index_dt"] = df_verr_ok["index_dt"].dt.date

In [ ]:
# Restrict interventions to rows that are actually part of the patient selection
keys = ["studyId_0888", "index_dt"]
df_verr_ok_pob = df_verr_ok.merge(df[keys], on=keys, how="inner")

print(df_verr_ok_pob.shape)
print(
    "Number of subjects who had a cardiologic intervention within a year after consulation: "
    f"{df_verr_ok_pob['studyId_0888'].nunique()}"
)

In [ ]:
# get all unique consultations in this selection
unique_combinations = df_verr_ok_pob[["studyId_0888", "index_dt"]].drop_duplicates()

ok_interventions = (
    df_verr_ok_pob.groupby(["studyId_0888", "index_dt"])["CODE_CODING_DISPLAY"]
    .agg(list)
    .rename("ok_interventions")
    .reset_index()
)

unique_combinations = unique_combinations.drop(columns="ok_interventions", errors="ignore").merge(
    ok_interventions, on=["studyId_0888", "index_dt"], how="left"
)

print(f"Total unique combinations: {len(unique_combinations)}")
unique_combinations["verr_ok"] = 1


### Load Radiology Interventions (verrichtingen)

In [ ]:
path_manager = PathManager_RES_Folder_UPOD()
path_verr_rad = path_manager.get_file_path("ARGUS25", "20250908", "verr_rad_20250527.sas7bdat")

df_verr_rad = pyreadstat.read_sas7bdat(path_verr_rad)[0]
df_verr_rad["studyId_0888"] = df_verr_rad["studyId_0888"].astype(int).astype(str)

# Drop rows where feature is empty
df_verr_rad = df_verr_rad[df_verr_rad["feature"] != ""]

In [ ]:
# Restrict radiology intervention data to entries based on patient selection
df_verr_rad["index_dt"] = df_verr_rad["index_dt"].dt.date
df_verr_rad_pob = df_verr_rad.merge(df[keys], on=keys, how="inner")

In [ ]:
# get unique studyId and index_dt combinations in df_verr_rad
unique_combinations_rad = df_verr_rad_pob[["studyId_0888", "index_dt", "feature"]].drop_duplicates()

print(f"Total unique combinations in verr_rad: {len(unique_combinations_rad)}")
unique_combinations_rad["verr_rad"] = 1

## Combine intervention and radiology procedure information

In [ ]:
# merge verr_ok and verr_rad on studyId and index_dt
df_rad_ok_verr = unique_combinations.merge(
    unique_combinations_rad,
    how="outer",
    on=["studyId_0888", "index_dt"],
)


In [ ]:
# make pivot table of feature column
df_rad_ok_verr_pivot = df_rad_ok_verr.pivot_table(
    index=["studyId_0888", "index_dt"],
    columns="feature",
    values="verr_rad",
    fill_value=0,
)


In [ ]:
# flatten the column names and add prefix
df_rad_ok_verr_pivot.columns = [f"{col}" for col in df_rad_ok_verr_pivot.columns]

# reset index to make it a regular dataframe
df_rad_ok_verr_pivot = df_rad_ok_verr_pivot.reset_index()

# convert feature columns to int type
feature_cols = df_rad_ok_verr_pivot.columns[2:]  # skip studyId_0888 and index_dt
df_rad_ok_verr_pivot[feature_cols] = df_rad_ok_verr_pivot[feature_cols].astype(int)


In [ ]:
keys = ["studyId_0888", "index_dt"]

df_rad_ok_verr_flat = unique_combinations[keys + ["verr_ok", "ok_interventions"]].merge(
    df_rad_ok_verr_pivot, on=keys, how="outer"
)
df_rad_ok_verr_flat[feature_cols] = df_rad_ok_verr_flat[feature_cols].fillna(0).astype(int)
df_rad_ok_verr_flat["verr_ok"] = df_rad_ok_verr_flat["verr_ok"].fillna(0).astype(int)
df_rad_ok_verr_flat["verr_rad"] = (df_rad_ok_verr_flat[feature_cols].sum(axis=1) > 0).astype(int)


df_rad_ok_verr_flat.info()

In [ ]:
# check if there are any duplicates in the studyId_0888 and index_dt combinations
duplicates = df_rad_ok_verr_flat[df_rad_ok_verr_flat.duplicated(subset=["studyId_0888", "index_dt"], keep=False)]
print(f"Number of duplicates: {len(duplicates)}")

No duplicates present

## Load CT rapports

First,load the original radio_verslag files to extract the index_dt

In [ ]:
path_rad_verslag = path_manager.get_file_path("ARGUS25", "20250908", "verr_rad_verslag_20250527.sas7bdat")

df_rad_verslag = pyreadstat.read_sas7bdat(path_rad_verslag, encoding="latin1")[0]
df_rad_verslag["studyId_0888"] = df_rad_verslag["studyId_0888"].astype(int).astype(str)

df_rad_verslag_index_dt = df_rad_verslag[["studyId_0888", "index_dt", "performedPeriod_start"]].drop_duplicates()

df_rad_verslag_index_dt.info()

In [ ]:
# Link each report to the patient's consultation, if the scan is within a year after the consulatation
visits = df[["studyId_0888", "index_dt"]].drop_duplicates()

links = df_rad_verslag[["studyId_0888", "performedPeriod_start"]].drop_duplicates()
links = links.merge(visits, on="studyId_0888", how="inner")
links["days"] = (links["performedPeriod_start"] - pd.to_datetime(links["index_dt"])).dt.days

print(links["days"].describe())  # check the range before applying the window
df_rad_verslag_index_dt = links[links["days"].between(0, 365)].drop(columns="days")

df_rad_verslag_index_dt.info()

Then load the NLP extracted CAD scores for all radio files. Note: these scores are extracted with the src/label/labeling_script.py file

In [ ]:
path_manager = PathManager_RES_Folder_UPOD()

path_labels = path_manager.get_file_path("ARGUS25", "radio_labels", "radio_label", "radio_labels.parquet")

df_labels = pd.read_parquet(path_labels)
df_labels["studyId_0888"] = df_labels["studyId_0888"].astype(int).astype(str)
df_labels["performedPeriod_start"] = pd.to_datetime(
    df_labels["performedPeriod_start"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)


In [ ]:
df_labels = df_labels.merge(
    df_rad_verslag_index_dt,
    how="inner",
    on=["studyId_0888", "performedPeriod_start"],
)

df_labels = df_labels.drop_duplicates()

df_labels = df_labels.drop(
    columns=[
        "verslagtekst",
        "cacs_percentile_2001",
        "cacs_percentile_2020",
        "performedPeriod_start",
    ]
)
df_labels = df_labels.drop_duplicates()

In [ ]:
keys = ["studyId_0888", "index_dt"]
print("rows with missing index_dt:", df_labels["index_dt"].isna().sum())

In [ ]:
def combine_labels(duplicate_group):
    """Combine duplicate labels by taking the highest non-null value.

    We prefer the highest non-null value to avoid underestimating risk and
    be on the safe side.
    """
    non_null = duplicate_group.dropna()
    if len(non_null) == 0:
        return np.nan
    return non_null.max()


df_labels_clean = (
    df_labels.groupby(["studyId_0888", "index_dt"])
    .agg(
        {
            "macro_stenosis": combine_labels,
            "macro_plaque": combine_labels,
            "cacs_mean": combine_labels,
            "macro_percentile_2001": combine_labels,
            "macro_percentile_2020": combine_labels,
            "macro_stenosegraad": combine_labels,
            "macro_cadrads": combine_labels,
            "macro_mesa": combine_labels,
            "MESA": combine_labels,
            "CADRADS": combine_labels,
        }
    )
    .reset_index()
)

print(f"Original: {len(df_labels)} rows")
print(f"After deduplication: {len(df_labels_clean)} rows")

duplicated = df_labels_clean[df_labels_clean.duplicated(subset=["studyId_0888", "index_dt"], keep=False)]
print(f"Number of duplicated rows (based on study ID and index dt): {len(duplicated)}")

df_labels = df_labels_clean.copy()
df_labels["studyId_0888"].nunique(), len(df_labels), df_labels.shape

# Combine df with CT report, intervention and radiology data

In [ ]:
ct_cols = ["macro_stenosis", "macro_stenosegraad", "macro_cadrads", "cacs_mean", "macro_mesa"]
has_ct_info = df_labels[ct_cols].notna().any(axis=1)

df_labels["CT_based_CAD"] = (
    (df_labels["macro_stenosis"] == 1)
    | (df_labels["macro_stenosegraad"] == 1)
    | (df_labels["macro_cadrads"] == 1)
    | (df_labels["cacs_mean"] > 400)
    | (df_labels["macro_mesa"] == 1)
).astype(int).where(has_ct_info)

df_labels["cacs_zero"] = (df_labels["cacs_mean"] == 0).astype(int).where(df_labels["cacs_mean"].notna())


In [ ]:
df_with_labels = df.merge(
    df_labels[["studyId_0888", "index_dt", "CT_based_CAD", "cacs_zero"]],
    how="left",
    on=["studyId_0888", "index_dt"],
)

In [ ]:
df_with_labels["studyId_0888"].nunique(), len(df_with_labels), df_with_labels.shape

In [ ]:
df_with_labels[["studyId_0888", "index_dt"]].drop_duplicates().shape  # unique combinations

In [ ]:
duplicated = df_with_labels[df_with_labels.duplicated(subset=["studyId_0888", "index_dt"], keep=False)]
print(f"Number of duplicated rows (based on study ID and index dt): {len(duplicated)}")


In [ ]:
df_labels_final = df_with_labels.merge(
    df_rad_ok_verr_flat,
    how="left",
    on=["studyId_0888", "index_dt"],
)

In [ ]:
df_labels_final["studyId_0888"].nunique(), len(df_labels_final), df_labels_final.shape

In [ ]:
df_labels_final["verr_ok"].value_counts(dropna=False)

In [ ]:
df_labels_final["verr_rad"].value_counts(dropna=False)

In [ ]:
df_labels_final["cacs_zero"].value_counts()  # 1 means no cacs, 0 means cacs > 0

# Label logic

In [ ]:
label_cols = [
    "CT_based_CAD",
    "cacs_zero",
    "CT Angio",
    "CT Thorax",
    "CT hart",
    "CT perfusie",
    "MR Angio",
    "MRI hart",
    "PET/CT",
    "SPECT",
    "verr_ok",
    "verr_rad",
]
df_label = df_labels_final[["studyId_0888", "gender", "Age", "index_dt"] + label_cols].copy()

In [ ]:
df_label["label"] = np.nan

If there is no mention of any diagnostic imagaging or treatment interventing related to CAD in the year after consultation, we assume that CAD can be safely excluded. Label = 0 

In [ ]:
df_label.loc[df_label[label_cols].isna().all(axis=1), "label"] = 0  # No mention of CAD or any diagnostic procedure
df_label["label"].value_counts()

If there is a CT but it mentioneds a CACS score of 0, we set the label to 0 and assume that CAD can be safely excluded. Note that this labelling is hierarchical. If there is another condition which indicates that CAD cannot be safely excluded, the label of zero will be overwritten, dispite the CACS of 0. 

In [ ]:
df_label.loc[df_label["cacs_zero"] == 1, "label"] = 0  # if cacs is zero, the label should be 0
df_label["label"].value_counts(dropna=False)

In [ ]:
df_label.loc[df_label["verr_ok"] == 1, "label"] = 1  # cardiologic intervention within a year after consultation
df_label["label"].value_counts(dropna=False)

If there is a chance that there is a risk of CAD we will overwrite the label=0

In [ ]:
df_label.loc[df_label["CT_based_CAD"] == 1, "label"] = 1  # CT-based CAD
df_label["label"].value_counts(dropna=False)

In [ ]:
df_label.loc[
    (df_label["MR Angio"] == 1) | (df_label["MRI hart"] == 1) | (df_label["PET/CT"] == 1) | (df_label["SPECT"] == 1),
    "label",
] = 1  # MR Angio, MRI hart, PET/CT, SPECT
df_label["label"].value_counts(dropna=False)

The remaining nan's are ambiguous. Lets look into the details some more:

In [ ]:
ambi_labels = df_label.loc[df_label["label"].isna()]

# merge with df_labels to get more details
ambi_labels = ambi_labels.merge(df_labels, how="left", on=["studyId_0888", "index_dt"])
ambi_labels.head()

In [ ]:
# Since we cannot say whether these records represent a very low risk for CAD or not,
# we drop the records with these ambiguous labels

df_label = df_label.dropna(subset=["label"])
df_label["label"].value_counts(dropna=False)

In [ ]:
print(f"Number of unique studyId_0888: {df_labels_final['studyId_0888'].nunique()}")
print(f"Number of entries: {len(df_labels_final)}")

# after label conversion:
print(f"Number of unique studyId_0888: {df_label['studyId_0888'].nunique()}")
print(f"Number of entries: {len(df_label)}")

In [ ]:
# drop labels and columns used for checking info (lab date and intervention list)
keys = ["studyId_0888", "index_dt"]
df_labels_final = df_label[keys + ["label"]].merge(
    df_labels_final.drop(columns=label_cols + ["ok_interventions"]),
    on=keys,
    how="left",
)

df_labels_final["label"] = df_labels_final["label"].astype(int)

print(f"Number of unique studyId_0888: {df_labels_final['studyId_0888'].nunique()}")
print(f"Number of entries: {len(df_labels_final)}")

In [ ]:
# Save label into original dataset incl preprocessed data and predictions
df_female = df_labels_final.loc[df_labels_final["gender"] == 1]
df_male = df_labels_final.loc[df_labels_final["gender"] == 0]

In [ ]:
# save to parquet
df_female.to_parquet(os.path.join(data_folder, "female_argus_data.parquet"))
df_male.to_parquet(os.path.join(data_folder, "male_argus_data.parquet"))